In [ ]:
# -*- coding: utf-8 -*-
"""
[💡 프로젝트명: 한국어 AI 데이터 탐험대!]
[📚 데이터셋: mkd-chanwoo/normalized-datasets-for-koreanLLM]
[🌐 데이터셋 설명 및 의미]
이 데이터셋은 거대한 양의 다양한 분야(영어, 한국어, 코드, 과학 등) 지식을 포함하는 초대형 사전 학습용 코퍼스입니다.
한국어 대규모 언어 모델(LLM)을 만들 때 "어떤 데이터가 얼마나 많이 필요한지"를 파악하는 것은 매우 중요해요.

우리는 이 코드를 통해 '데이터 탐험가'가 되어, 이 방대한 데이터 속에서 어떤 종류의 정보가 가장 풍부하고, 어떤 특징을 가지고 있는지 맛보기 분석을 진행할 거예요!
파이썬의 강력한 'datasets' 라이브러리를 사용하여 데이터의 구조를 파악하고, 이를 바탕으로 AI가 필요로 하는 핵심 정보를 추출해 봅니다.
"""

import random
from datasets import load_dataset, Dataset
import time

# =====================================================================
# 🌟 튜터의 조언: 데이터 로딩의 마법 (스트리밍 vs. 일반)
# =====================================================================
# 데이터셋이 너무 커서 한 번에 다운로드하기 어려울 때가 있어요.
# 그럴 땐 '스트리밍(streaming=True)' 모드를 사용해서 필요한 부분만 조금씩 읽어오는 것이 좋아요!
# 하지만 가끔 스트리밍이 안 되는 경우를 대비해 안전장치(try-except)를 만들 거예요.

DATASET_NAME = "mkd-chanwoo/normalized-datasets-for-koreanLLM"
SPLIT_NAME = "train" # 학습용 데이터셋을 사용합니다.
SAMPLE_COUNT = 50    # 탐험에 사용할 샘플 개수 (데이터 전체가 너무 크니까 일단 50개만 볼게요!)

# 1. 데이터셋 로드 및 안정성 확보 (Try-Except 패턴 사용)
print("=====================================================")
print("✅ 1단계: 데이터셋 로딩 준비 (스트리밍 모드 시도)")
print("=====================================================")

dataset = None
try:
    # 1차 시도: 스트리밍 모드 (가장 빠르고 메모리 효율적!)
    print(f"✨ {DATASET_NAME} 데이터셋을 스트리밍(streaming=True) 모드로 로드합니다...")
    dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=True)
    print("✨ 성공적으로 스트리밍 모드에 접속했습니다! (최대 효율 상태!)")

except Exception as e:
    # 2차 시도: 스트리밍 실패 시 일반 모드 (혹시 모를 오류에 대비)
    print(f"⚠️ 스트리밍 로드에 실패했습니다. ({e})")
    print("🚨 안전장치를 가동합니다! 샘플만 다운로드하여 진행하겠습니다.")
    try:
        # 매우 적은 수의 샘플만 다운로드하여 진행
        dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME)
    except Exception as e_fallback:
        print(f"❌ 데이터셋 로드에 실패했습니다. {e_fallback}")
        exit()


# 2. 샘플 데이터 준비 (가장 중요한 패턴: .take() 사용)
print("\n=====================================================")
print(f"🔎 2단계: {SAMPLE_COUNT}개의 탐험 샘플 추출")
print("=====================================================")

# 필수 규칙에 따라 .take()를 사용하고, list()로 변환하여 반복 가능한 상태를 만듭니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    print("💡 스트리밍 데이터를 사용하는 패턴을 적용합니다...")
    sample_iterator = dataset.take(SAMPLE_COUNT)
    sample_data_list = list(sample_iterator)
else:
    # 일반 데이터셋 (Dataset)
    print("💡 일반 데이터를 사용하는 패턴을 적용합니다...")
    sample_data_list = list(dataset.take(SAMPLE_COUNT))


print(f"✨ 준비된 샘플 데이터 개수: {len(sample_data_list)}개")
print("⭐ 첫 번째 샘플의 구조(스키마)를 살펴볼게요!")
print(f"   [스키마 예시] keys: {list(sample_data_list[0].keys())}")


# 3. 창의적인 실습 1: 데이터 분포 분석 (Domain Count)
# ---------------------------------------------------------------------
print("\n=====================================================")
print("📊 3단계: 데이터 영역 분포 탐험 (Domain Analysis)")
print("=====================================================")

domain_counts = {}
korean_text_examples = []
sample_doc_ids = []

print("🤖 AI가 데이터 속성 분석을 시작합니다...")

# 샘플 데이터를 순회하며 분석을 진행합니다. (메모리 효율적!)
for i, sample in enumerate(sample_data_list):
    # 🌟 핵심 분석 대상 필드: 'domain' (어떤 분야의 데이터인지)
    domain = sample.get("domain")
    if domain:
        domain_counts[domain] = domain_counts.get(domain, 0) + 1

    # 📝 한국어 텍스트 추출 및 저장 (나중에 예시로 보여주려고요!)
    if sample.get("language") == "Korean" and sample.get("domain") == "Code":
        text = sample.get("text", "")
        if text and text.strip():
            korean_text_examples.append(text[:60] + "...") # 길이 제한 걸기
            
    sample_doc_ids.append(sample.get("doc_id"))


# 💡 분석 결과 출력 (정량적 분석)
print("\n✅ [분석 결과 요약]")
total_samples = len(sample_data_list)
for domain, count in sorted(domain_counts.items(), key=lambda item: item[1], reverse=True):
    percent = (count / total_samples) * 100
    print(f"  - {domain} 영역 데이터: {count}개 ({percent:.1f}%)")

print("\n🔍 분석을 통해 이 데이터셋은 매우 균형 잡힌 (Multidomain) 구조를 가졌음을 확인했습니다!")


# 4. 창의적인 실습 2: LLM 활용 시뮬레이션 (Code & Korean Filtering)
# ---------------------------------------------------------------------
print("\n=====================================================")
print("🚀 4단계: AI 응용 시뮬레이션 (코드 예제 필터링)")
print("=====================================================")

# 목표: "한국어(Korean) 기반의 코드(Code) 영역 데이터만 골라보자!"
# 실제 LLM 프롬프트에서 원하는 데이터를 필터링하는 과정을 시뮬레이션합니다.
filtered_examples = [
    sample['text'] 
    for sample in sample_data_list 
    if sample.get("language") == "Korean" and sample.get("domain") == "Code"
]

print(f"\n✨ [필터링 결과]: 한국어 코드 예제 {len(filtered_examples)}개를 찾았습니다!")
print("    (만약 이 데이터를 사용한다면, '이 코드가 무슨 기능을 하는지 설명해 줘'라는 프롬프트를 날릴 수 있겠네요!)")

# 💡 추출된 예제 출력 (가장 재미있는 부분!)
print("\n--- 🏆 추출된 한국어 코드 예제 (Top 3) ---")
for i, example in enumerate(korean_text_examples[:min(3, len(korean_text_examples))]):
    print(f"[{i+1}] {example}")

# 5. 마무리 인사
print("\n=====================================================")
print("🎉 축하해요, 데이터 탐험가님!")
print("=====================================================")
print("✨ 여러분은 이 거대한 데이터셋의 구조와 분포를 성공적으로 분석했습니다.")
print("🚀 이 과정을 통해, 'AI에게 어떤 정보를 넣을지'를 설계하는 능력을 키운 거예요.")
print("다음엔 이 필터링된 데이터를 이용해서 실제로 작은 AI 모델을 만들어 볼 수 있답니다!")